In [1]:
import pandas as pd
import numpy as np
import timeit
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler

print(f'pandas : {pd.__version__}')
print(f'numpy  : {np.__version__}')
print('Всі бібліотеки імпортовано ✓')

pandas : 3.0.1
numpy  : 2.4.2
Всі бібліотеки імпортовано ✓


In [2]:
import urllib.request
import zipfile
import os

url = 'https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip'
zip_path = 'dataset.zip'

print('Завантаження...')
urllib.request.urlretrieve(url, zip_path)

print('Розпакування...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('.')

print('Файли в папці:')
for f in os.listdir('.'):
    print(f'  {f}')

Завантаження...
Розпакування...
Файли в папці:
  .ipynb_checkpoints
  data
  dataset.zip
  household_power_consumption.txt
  laba2part1.ipynb
  laba2part2.ipynb
  vhi_analysis


In [3]:
df = pd.read_csv(
    'household_power_consumption.txt',
    sep=';',
    na_values=['?'],
    low_memory=False
)

print(f'Розмір: {df.shape[0]:,} рядків × {df.shape[1]} стовпців')
print(df.dtypes)
df.head()

Розмір: 2,075,259 рядків × 9 стовпців
Date                         str
Time                         str
Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [4]:
df['Datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%d/%m/%Y %H:%M:%S'
)

df.drop(columns=['Date', 'Time'], inplace=True)
df.set_index('Datetime', inplace=True)

print(f'Діапазон: {df.index.min()} → {df.index.max()}')
df.head()

Діапазон: 2006-12-16 17:24:00 → 2010-11-26 21:02:00


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [5]:
numeric_cols = [
    'Global_active_power', 'Global_reactive_power',
    'Voltage', 'Global_intensity',
    'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
]

df[numeric_cols] = df[numeric_cols].astype(float)
print(df.dtypes)

Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object


In [6]:

missing_report = pd.DataFrame({
    'Пропущено (шт)': df.isnull().sum(),
    'Пропущено (%)' : (df.isnull().mean() * 100).round(2)
})
print(missing_report)
print(f'\nЗагалом: {df.isnull().sum().sum():,} пропусків')

                       Пропущено (шт)  Пропущено (%)
Global_active_power             25979           1.25
Global_reactive_power           25979           1.25
Voltage                         25979           1.25
Global_intensity                25979           1.25
Sub_metering_1                  25979           1.25
Sub_metering_2                  25979           1.25
Sub_metering_3                  25979           1.25

Загалом: 181,853 пропусків


In [7]:
df.ffill(inplace=True)
df.bfill(inplace=True)  

print(f'Залишилось пропусків: {df.isnull().sum().sum()}')

Залишилось пропусків: 0


In [8]:
df['Hour'] = df.index.hour

print(f'Фінальний розмір: {df.shape}')
df.describe().round(3)

Фінальний розмір: (2075259, 8)


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Hour
count,2075259.000,2075259.000,2075259.000,2075259.000,2075259.000,2075259.000,2075259.000,2075259.000
mean,1.086,0.123,240.842,4.604,1.111,1.288,6.417,11.501
std,1.053,0.113,3.236,4.427,6.116,5.787,8.420,6.922
min,0.076,0.000,223.200,0.200,0.000,0.000,0.000,0.000
25%,0.308,0.048,239.000,1.400,0.000,0.000,0.000,6.000
50%,0.598,0.100,241.020,2.600,0.000,0.000,1.000,12.000
75%,1.524,0.194,242.870,6.400,0.000,1.000,17.000,18.000
max,11.122,1.390,254.150,48.400,88.000,80.000,31.000,23.000


In [9]:
def filter_high_power(df, threshold=5.0):
    """
    Повертає всі записи де Global_active_power > threshold кВт.
    """
    return df[df['Global_active_power'] > threshold].copy()


time_31 = timeit.timeit(lambda: filter_high_power(df), number=100) / 100
result_31 = filter_high_power(df)

print('=== GAP > 5 кВт ===')
print(f'Час виконання    : {time_31*1000:.3f} мс')
print(f'Знайдено записів : {len(result_31):,}  ({len(result_31)/len(df)*100:.2f}% від усіх)')
print(f'\nСтатистика GAP у вибірці:')
print(result_31['Global_active_power'].describe().round(3))
result_31[['Global_active_power', 'Global_intensity', 'Voltage']].head(10)

=== GAP > 5 кВт ===
Час виконання    : 3.891 мс
Знайдено записів : 17,550  (0.85% від усіх)

Статистика GAP у вибірці:
count    17550.000
mean         5.853
std          0.779
min          5.002
25%          5.274
50%          5.648
75%          6.172
max         11.122
Name: Global_active_power, dtype: float64


,Global_active_power,Global_intensity,Voltage
Datetime,,,
2006-12-16 17:25:00,5.360,23.0,233.63
2006-12-16 17:26:00,5.374,23.0,233.29
2006-12-16 17:27:00,5.388,23.0,233.74
2006-12-16 17:35:00,5.412,23.2,232.78
2006-12-16 17:36:00,5.224,22.4,232.99
2006-12-16 17:37:00,5.268,22.6,232.91
2006-12-16 17:44:00,5.894,25.4,232.69
2006-12-16 17:45:00,7.706,33.2,230.98
2006-12-16 17:46:00,7.026,30.6,232.21


In [10]:
def filter_current_and_compare(df, i_min=19, i_max=20):
    """
    Крок 1 — Global_intensity ∈ [i_min, i_max].
    Крок 2 — з відібраних: Sub_metering_2 > Sub_metering_3.
    """
    step1 = df[
        (df['Global_intensity'] >= i_min) &
        (df['Global_intensity'] <= i_max)
    ].copy()

    step2 = step1[step1['Sub_metering_2'] > step1['Sub_metering_3']].copy()

    return step1, step2


time_32 = timeit.timeit(lambda: filter_current_and_compare(df), number=100) / 100
step1_32, result_32 = filter_current_and_compare(df)

print('=== Струм 19–20 А → SM2 > SM3 ===')
print(f'Час виконання                : {time_32*1000:.3f} мс')
print(f'Крок 1 (струм 19–20 А)      : {len(step1_32):,} записів')
print(f'Крок 2 (SM2 > SM3)          : {len(result_32):,} записів')
print(f'\nСередні значення у вибірці:')
print(f'  SM2 (пральна/холод.) : {result_32["Sub_metering_2"].mean():.2f} Вт·год')
print(f'  SM3 (бойлер/кондиц.) : {result_32["Sub_metering_3"].mean():.2f} Вт·год')
result_32[['Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].head(10)

=== Струм 19–20 А → SM2 > SM3 ===
Час виконання                : 5.935 мс
Крок 1 (струм 19–20 А)      : 7,022 записів
Крок 2 (SM2 > SM3)          : 2,509 записів

Середні значення у вибірці:
  SM2 (пральна/холод.) : 35.75 Вт·год
  SM3 (бойлер/кондиц.) : 11.39 Вт·год


,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,
2006-12-16 18:09:00,19.0,0.0,37.0,16.0
2006-12-17 01:04:00,19.6,0.0,13.0,0.0
2006-12-17 01:08:00,19.6,0.0,27.0,0.0
2006-12-17 01:19:00,19.4,0.0,36.0,0.0
2006-12-17 01:20:00,19.4,0.0,35.0,0.0
2006-12-17 01:21:00,19.4,0.0,36.0,0.0
2006-12-17 01:52:00,19.2,0.0,37.0,0.0
2006-12-17 09:08:00,20.0,0.0,38.0,0.0
2006-12-17 09:09:00,19.0,0.0,38.0,0.0


In [11]:
def random_sample_means(df, n=500_000, random_state=42):
    """
    Обирає n записів без повторів,
    повертає вибірку та середні по трьох групах SM.
    """
    if n > len(df):
        raise ValueError(f'n={n:,} > розміру датасету ({len(df):,})')

    sample = df.sample(n=n, replace=False, random_state=random_state)

    means = {
        'SM1 — Кухня'               : sample['Sub_metering_1'].mean(),
        'SM2 — Пральна/холодильник' : sample['Sub_metering_2'].mean(),
        'SM3 — Бойлер/кондиціонер'  : sample['Sub_metering_3'].mean(),
    }
    return sample, means


time_33 = timeit.timeit(lambda: random_sample_means(df), number=10) / 10
sample_33, means_33 = random_sample_means(df)

print('=== Рандом 500 000 записів ===')
print(f'Час виконання  : {time_33*1000:.3f} мс')
print(f'Розмір вибірки : {len(sample_33):,}')
print(f'Дублів         : {sample_33.index.duplicated().sum()}  (має бути 0)')
print(f'\nСередні значення груп споживання (Вт·год):')
for name, val in means_33.items():
    print(f'  {name}: {val:.4f}')

=== Рандом 500 000 записів ===
Час виконання  : 102.664 мс
Розмір вибірки : 500,000
Дублів         : 0  (має бути 0)

Середні значення груп споживання (Вт·год):
  SM1 — Кухня: 1.1122
  SM2 — Пральна/холодильник: 1.2895
  SM3 — Бойлер/кондиціонер: 6.4244


In [12]:
def complex_evening_filter(df):
    """
    Крок 1 — Hour >= 18 та GAP > 6 кВт.
    Крок 2 — SM2 є найбільшою серед SM1, SM2, SM3.
    Крок 3 — кожен 3-й з 1-ї половини + кожен 4-й з 2-ї.
    """
    # Крок 1
    step1 = df[
        (df['Hour'] >= 18) &
        (df['Global_active_power'] > 6)
    ].copy()

    # Крок 2: idxmax по axis=1 повертає ім'я найбільшого стовпця в рядку
    sub_cols = ['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    step2 = step1[
        step1[sub_cols].idxmax(axis=1) == 'Sub_metering_2'
    ].copy()

    # Крок 3
    mid         = len(step2) // 2
    first_half  = step2.iloc[:mid:3]   # кожен 3-й
    second_half = step2.iloc[mid::4]   # кожен 4-й
    result      = pd.concat([first_half, second_half]).sort_index()

    print(f'Крок 1 (після 18:00, > 6 кВт)   : {len(step1):>8,}')
    print(f'Крок 2 (SM2 найбільша)           : {len(step2):>8,}')
    print(f'Крок 3a (кожен 3-й з 1-ї пол.)  : {len(first_half):>8,}')
    print(f'Крок 3b (кожен 4-й з 2-ї пол.)  : {len(second_half):>8,}')
    print(f'Фінал                            : {len(result):>8,}')

    return result


time_34 = timeit.timeit(lambda: complex_evening_filter(df), number=50) / 50
result_34 = complex_evening_filter(df)

print(f'\nЧас виконання: {time_34*1000:.3f} мс')
result_34[['Global_active_power', 'Hour', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].head(10)

Крок 1 (після 18:00, > 6 кВт)   :    2,882
Крок 2 (SM2 найбільша)           :    1,062
Крок 3a (кожен 3-й з 1-ї пол.)  :      177
Крок 3b (кожен 4-й з 2-ї пол.)  :      133
Фінал                            :      310
Крок 1 (після 18:00, > 6 кВт)   :    2,882
Крок 2 (SM2 найбільша)           :    1,062
Крок 3a (кожен 3-й з 1-ї пол.)  :      177
Крок 3b (кожен 4-й з 2-ї пол.)  :      133
Фінал                            :      310
Крок 1 (після 18:00, > 6 кВт)   :    2,882
Крок 2 (SM2 найбільша)           :    1,062
Крок 3a (кожен 3-й з 1-ї пол.)  :      177
Крок 3b (кожен 4-й з 2-ї пол.)  :      133
Фінал                            :      310
Крок 1 (після 18:00, > 6 кВт)   :    2,882
Крок 2 (SM2 найбільша)           :    1,062
Крок 3a (кожен 3-й з 1-ї пол.)  :      177
Крок 3b (кожен 4-й з 2-ї пол.)  :      133
Фінал                            :      310
Крок 1 (після 18:00, > 6 кВт)   :    2,882
Крок 2 (SM2 найбільша)           :    1,062
Крок 3a (кожен 3-й з 1-ї пол.)  :      177
Кр

,Global_active_power,Hour,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,
2006-12-16 18:05:00,6.052,18,0.0,37.0,17.0
2006-12-16 18:08:00,6.308,18,0.0,36.0,17.0
2006-12-28 20:58:00,6.386,20,1.0,36.0,17.0
2006-12-28 21:02:00,8.088,21,1.0,72.0,17.0
2006-12-28 21:05:00,7.230,21,1.0,73.0,17.0
2006-12-28 21:08:00,7.352,21,1.0,73.0,17.0
2006-12-28 21:11:00,9.048,21,34.0,71.0,16.0
2006-12-28 21:14:00,9.118,21,36.0,72.0,16.0
2006-12-28 21:17:00,7.040,21,37.0,38.0,17.0


In [13]:
pd.DataFrame({
    'Завдання': [
        '3.1 — GAP > 5 кВт',
        '3.2 — Струм 19–20А, SM2 > SM3',
        '3.3 — Рандом 500k',
        '3.4 — Вечір > 6кВт, SM2 домінує',
    ],
    'Час (мс)': [
        round(time_31 * 1000, 3),
        round(time_32 * 1000, 3),
        round(time_33 * 1000, 3),
        round(time_34 * 1000, 3),
    ],
    'Записів у результаті': [
        len(result_31), len(result_32), len(sample_33), len(result_34)
    ]
})

,Завдання,Час (мс),Записів у результаті
0,3.1 — GAP > 5 кВт,3.891,17550
1,"3.2 — Струм 19–20А, SM2 > SM3",5.935,2509
2,3.3 — Рандом 500k,102.664,500000
3,"3.4 — Вечір > 6кВт, SM2 домінує",5.976,310


In [14]:
def normalize_minmax(df):
    """
    Min-Max: x_norm = (x - x_min) / (x_max - x_min)
    Результат: всі значення у [0, 1].
    """
    cols = [
        'Global_active_power', 'Global_reactive_power', 'Voltage',
        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
    ]
    scaler    = MinMaxScaler()
    df_norm   = df.copy()
    df_norm[cols] = scaler.fit_transform(df[cols])
    return df_norm, scaler


df_normalized, minmax_scaler = normalize_minmax(df)

# Перевірка: min=0, max=1
check_cols = ['Global_active_power', 'Voltage', 'Global_intensity']
pd.DataFrame({
    'min' : df_normalized[check_cols].min(),
    'max' : df_normalized[check_cols].max(),
    'mean': df_normalized[check_cols].mean().round(4),
})

,min,max,mean
Global_active_power,0.0,1.0,0.0915
Voltage,0.0,1.0,0.5700
Global_intensity,0.0,1.0,0.0914


In [15]:
def standardize_zscore(df):
    """
    Z-score: x_std = (x - mean) / std
    Результат: середнє ≈ 0, стандартне відхилення ≈ 1.
    """
    cols = [
        'Global_active_power', 'Global_reactive_power', 'Voltage',
        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
    ]
    scaler   = StandardScaler()
    df_std   = df.copy()
    df_std[cols] = scaler.fit_transform(df[cols])
    return df_std, scaler


df_standardized, std_scaler = standardize_zscore(df)

# Перевірка: mean≈0, std≈1
check_cols = ['Global_active_power', 'Voltage', 'Global_intensity']
pd.DataFrame({
    'mean': df_standardized[check_cols].mean().round(6),
    'std' : df_standardized[check_cols].std().round(4),
    'min' : df_standardized[check_cols].min().round(3),
    'max' : df_standardized[check_cols].max().round(3),
})

,mean,std,min,max
Global_active_power,-0.0,1.0,-0.959,9.529
Voltage,0.0,1.0,-5.452,4.113
Global_intensity,0.0,1.0,-0.995,9.893


In [16]:
col = 'Global_active_power'

pd.DataFrame({
    'Оригінал'     : [df[col].min(),             df[col].max(),             df[col].mean(),             df[col].std()],
    'Min-Max [0,1]': [df_normalized[col].min(),  df_normalized[col].max(),  df_normalized[col].mean(),  df_normalized[col].std()],
    'Z-score'      : [df_standardized[col].min(),df_standardized[col].max(),df_standardized[col].mean(),df_standardized[col].std()],
}, index=['min', 'max', 'mean', 'std']).round(4)

,Оригінал,"Min-Max [0,1]",Z-score
min,0.0760,0.0000,-0.9592
max,11.1220,1.0000,9.5291
mean,1.0862,0.0915,-0.0000
std,1.0532,0.0953,1.0000


In [17]:
def interpret_r(r):
    a = abs(r)
    direction = 'позитивний' if r >= 0 else 'негативний'
    if   a >= 0.9: strength = 'дуже сильний'
    elif a >= 0.7: strength = 'сильний'
    elif a >= 0.5: strength = 'помірний'
    elif a >= 0.3: strength = 'слабкий'
    else:          strength = 'дуже слабкий / відсутній'
    return f'{strength} {direction}'


def compute_correlations(df, col1, col2):
    """
    Обчислює Пірсона та Спірмена між двома числовими стовпцями.
    p-value < 0.05 → результат статистично значущий.
    """
    data = df[[col1, col2]].dropna()

    pearson_r,  pearson_p  = stats.pearsonr(data[col1],  data[col2])
    spearman_r, spearman_p = stats.spearmanr(data[col1], data[col2])

    print(f'=== [{col1}]  vs  [{col2}] ===')
    print(f'Записів: {len(data):,}')
    print()
    print(f'Пірсон  r  = {pearson_r:+.6f}   p = {pearson_p:.2e}   → {interpret_r(pearson_r)}')
    print(f'Спірмен ρ  = {spearman_r:+.6f}   p = {spearman_p:.2e}   → {interpret_r(spearman_r)}')

    return pearson_r, spearman_r

In [18]:
p1, s1 = compute_correlations(df, 'Global_active_power', 'Global_intensity')

=== [Global_active_power]  vs  [Global_intensity] ===
Записів: 2,075,259

Пірсон  r  = +0.998884   p = 0.00e+00   → дуже сильний позитивний
Спірмен ρ  = +0.995426   p = 0.00e+00   → дуже сильний позитивний


In [19]:

p2, s2 = compute_correlations(df, 'Sub_metering_1', 'Sub_metering_2')

=== [Sub_metering_1]  vs  [Sub_metering_2] ===
Записів: 2,075,259

Пірсон  r  = +0.055102   p = 0.00e+00   → дуже слабкий / відсутній позитивний
Спірмен ρ  = +0.065426   p = 0.00e+00   → дуже слабкий / відсутній позитивний


In [20]:
numeric_only = df.select_dtypes(include=[np.number]).drop(columns=['Hour'])
numeric_only.corr(method='pearson').round(3)

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Global_active_power,1.000,0.246,-0.396,0.999,0.484,0.435,0.640
Global_reactive_power,0.246,1.000,-0.112,0.266,0.123,0.139,0.091
Voltage,-0.396,-0.112,1.000,-0.407,-0.194,-0.166,-0.260
Global_intensity,0.999,0.266,-0.407,1.000,0.489,0.440,0.628
Sub_metering_1,0.484,0.123,-0.194,0.489,1.000,0.055,0.104
Sub_metering_2,0.435,0.139,-0.166,0.440,0.055,1.000,0.082
Sub_metering_3,0.640,0.091,-0.260,0.628,0.104,0.082,1.000


In [21]:
def add_time_of_day(df):
    """
    Додає стовпець Time_of_Day на основі години:
      Ніч 0–5, Ранок 6–11, День 12–17, Вечір 18–23
    """
    def label(h):
        if   6  <= h < 12: return 'Ранок'
        elif 12 <= h < 18: return 'День'
        elif 18 <= h < 24: return 'Вечір'
        else:               return 'Ніч'

    df = df.copy()
    df['Time_of_Day'] = df['Hour'].apply(label).astype('category')
    return df


df_cat = add_time_of_day(df)

print('Розподіл категорій:')
counts = df_cat['Time_of_Day'].value_counts()
for cat, cnt in counts.items():
    bar = '█' * int(cnt / len(df_cat) * 50)
    print(f'  {cat:<7}: {cnt:>8,}  {bar}')

df_cat[['Hour', 'Time_of_Day']].head(12)

Розподіл категорій:
  Вечір  :  518,943  ████████████
  День   :  518,796  ████████████
  Ніч    :  518,760  ████████████
  Ранок  :  518,760  ████████████


,Hour,Time_of_Day
Datetime,,
2006-12-16 17:24:00,17,День
2006-12-16 17:25:00,17,День
2006-12-16 17:26:00,17,День
2006-12-16 17:27:00,17,День
2006-12-16 17:28:00,17,День
2006-12-16 17:29:00,17,День
2006-12-16 17:30:00,17,День
2006-12-16 17:31:00,17,День
2006-12-16 17:32:00,17,День


In [22]:
def one_hot_encode(df, col='Time_of_Day'):
    """
    pd.get_dummies — вбудований метод OHE у pandas.
    drop_first=False → зберігаємо всі категорії (K стовпців).
    dtype=int        → значення 0/1 замість True/False.
    """
    print(f'До OHE  : {df.shape[1]} стовпців')
    df_enc = pd.get_dummies(df, columns=[col], drop_first=False, dtype=int)
    print(f'Після   : {df_enc.shape[1]} стовпців')
    new_cols = [c for c in df_enc.columns if col in c]
    print(f'Нові    : {new_cols}')
    return df_enc


df_ohe = one_hot_encode(df_cat)

# Перевірка: сума по рядку = 1 (кожен запис в одній категорії)
ohe_cols = [c for c in df_ohe.columns if 'Time_of_Day' in c]
assert df_ohe[ohe_cols].sum(axis=1).eq(1).all(), 'Помилка!'
print('\nСума по рядку завжди = 1 ✓')

df_ohe[['Hour'] + ohe_cols].head(15)

До OHE  : 9 стовпців
Після   : 12 стовпців
Нові    : ['Time_of_Day_Вечір', 'Time_of_Day_День', 'Time_of_Day_Ніч', 'Time_of_Day_Ранок']

Сума по рядку завжди = 1 ✓


,Hour,Time_of_Day_Вечір,Time_of_Day_День,Time_of_Day_Ніч,Time_of_Day_Ранок
Datetime,,,,,
2006-12-16 17:24:00,17,0,1,0,0
2006-12-16 17:25:00,17,0,1,0,0
2006-12-16 17:26:00,17,0,1,0,0
2006-12-16 17:27:00,17,0,1,0,0
2006-12-16 17:28:00,17,0,1,0,0
2006-12-16 17:29:00,17,0,1,0,0
2006-12-16 17:30:00,17,0,1,0,0
2006-12-16 17:31:00,17,0,1,0,0
2006-12-16 17:32:00,17,0,1,0,0


In [23]:
demo = df_cat[['Time_of_Day']].head(6)

print('── drop_first=False (всі 4 стовпці) ──')
print(pd.get_dummies(demo, dtype=int).to_string())

print('\n── drop_first=True (3 стовпці, без надлишкового) ──')
print(pd.get_dummies(demo, drop_first=True, dtype=int).to_string())

print('\ndrop_first=True → для лінійних моделей (уникаємо мультиколінеарності)')
print('drop_first=False → для дерев, нейромереж, або коли потрібна інтерпретованість')

── drop_first=False (всі 4 стовпці) ──
                     Time_of_Day_Вечір  Time_of_Day_День  Time_of_Day_Ніч  Time_of_Day_Ранок
Datetime                                                                                    
2006-12-16 17:24:00                  0                 1                0                  0
2006-12-16 17:25:00                  0                 1                0                  0
2006-12-16 17:26:00                  0                 1                0                  0
2006-12-16 17:27:00                  0                 1                0                  0
2006-12-16 17:28:00                  0                 1                0                  0
2006-12-16 17:29:00                  0                 1                0                  0

── drop_first=True (3 стовпці, без надлишкового) ──
                     Time_of_Day_День  Time_of_Day_Ніч  Time_of_Day_Ранок
Datetime                                                                 
2006-12-16 17:24